# CALM — Notebook kiểm thử (`test.ipynb`)

Chạy lần lượt các cell để kiểm tra **toàn bộ agent** và **luồng end-to-end**.

**Yêu cầu:** Python ≥ 3.10, `pip install -e .`, file `.env` có ít nhất `OPENAI_API_KEY` hoặc `OPENROUTER_API_KEY`. Một số cell gọi GEE/Copernicus nếu bật trong `.env`.

Kernel nên đặt **working directory** là thư mục repo (có `pyproject.toml` và `src/calm`).


In [ ]:
# 0 — Setup: tìm repo CALM, thêm src vào path, load .env
from __future__ import annotations

import os
import sys
from pathlib import Path


def discover_calm_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "calm").is_dir():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy CALM: mở notebook từ thư mục chứa pyproject.toml hoặc `cd` vào repo rồi chạy lại."
    )


CALM_DIR = discover_calm_dir()
CALM_SRC = CALM_DIR / "src"
if str(CALM_SRC) not in sys.path:
    sys.path.insert(0, str(CALM_SRC))

from calm.utils.env_loader import load_env

load_env(CALM_DIR / ".env")
load_env(CALM_DIR.parent / ".env")

print("CALM_DIR =", CALM_DIR)
print("OPENAI_API_KEY:", "set" if os.environ.get("OPENAI_API_KEY") else "missing")
print("OPENROUTER_API_KEY:", "set" if os.environ.get("OPENROUTER_API_KEY") else "missing")


In [ ]:
# 1 — LLM dùng chung cho mọi agent
import os


def build_llm():
    if os.environ.get("OPENROUTER_API_KEY"):
        from langchain_openrouter import ChatOpenRouter
        return ChatOpenRouter(
            model=os.environ.get("OPENROUTER_MODEL", "openai/gpt-4o-mini"),
            api_key=os.environ["OPENROUTER_API_KEY"],
            temperature=0.0,
        )
    if os.environ.get("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(
            model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
            openai_api_key=os.environ["OPENAI_API_KEY"],
            temperature=0.0,
        )
    raise RuntimeError("Cần OPENAI_API_KEY hoặc OPENROUTER_API_KEY trong .env")


llm = build_llm()
print("LLM OK:", type(llm).__name__)


In [ ]:
# 2 — Orchestrator factory: một lần khởi tạo đủ agent + tool (giống production)
from calm.memory.chroma_store import ChromaMemoryStore
from calm.orchestrator import CALMOrchestrator

persist = str(CALM_DIR / ".chroma_test_notebook")
memory_store = ChromaMemoryStore(
    collection_name="calm_test_nb",
    persist_directory=persist,
    use_openai_embeddings=bool(os.environ.get("OPENAI_API_KEY")),
)

config = {
    "planner_n_max": 2,
    "qa_n_max": 2,
    "rsen_k": 2,
}

orch = CALMOrchestrator.from_llm(
    llm=llm,
    memory_store=memory_store,
    tools={},
    config=config,
)

# Tham chiếu nhanh tới từng agent (được gắn trong orchestrator)
assert orch.planner is not None
assert orch.router_agent is not None
assert orch.data_agent is not None
assert orch.qa_agent is not None
assert orch.prediction_agent is not None
assert orch.rsen is not None
assert orch.executor is not None
assert orch.memory_agent is not None

print("CALMOrchestrator.from_llm OK — đủ agent.")


## 3. QueryNormalizerAgent

Chuẩn hoá query thành context JSON (không bắt buộc trong luồng `run` hiện tại nhưng có trong codebase).


In [ ]:
from calm.agents.query_normalizer_agent import QueryNormalizerAgent
from calm.tools.geocoding import GeocodingTool
from calm.tools.safety_check import SafetyChecker

geocoder = GeocodingTool(safety_checker=SafetyChecker(llm=llm), config={})
normalizer = QueryNormalizerAgent(geocoder=geocoder)

qctx = normalizer.normalize("Predict wildfire risk for Central Valley California next 7 days")
print("keys:", sorted(k for k in qctx.keys() if not k.startswith("_"))[:12], "...")
print("location_resolved:", qctx.get("location_resolved"))
print("time_range:", qctx.get("time_range"))


## 4. PlanningAgent

URSA: Generator → Reflector → Formalizer → `final_output` (list steps).


In [ ]:
plan_state = orch.planner.invoke("What environmental factors affect Amazon wildfires?")
plan_steps = plan_state.get("final_output") or []
print("N steps:", len(plan_steps))
if plan_steps:
    print("First step keys:", list(plan_steps[0].keys()))
    print("First agent/action:", plan_steps[0].get("agent"), "|", plan_steps[0].get("action")[:60] if plan_steps[0].get("action") else "")
if plan_state.get("error"):
    print("Planner warning/error:", plan_state.get("error"))


## 5. RouterAgent

`task_type`: `qa` | `prediction` | `hybrid` (orchestrator sẽ map hybrid → prediction pipeline).


In [ ]:
query_demo = "Forecast wildfire likelihood in Northern California over the next 14 days"
routing = orch.router_agent.route(query_demo, plan_steps)
print("task_type:", routing.task_type)
print("confidence:", routing.confidence)
print("artifacts:", routing.required_artifacts[:5])
print("reasoning:", routing.reasoning[:120] if routing.reasoning else "")


## 6. DataKnowledgeAgent — `retrieve`

Có thể gọi GEE/Copernicus/Web tùy `.env`. Trả về `retrieved_data` + extracted knowledge.


In [ ]:
from calm.utils.time_utils import resolve_time_range

params = {
    "location": "California, USA",
    "time_range": resolve_time_range("next 7 days", default_today=True),
}

data_res = orch.data_agent.retrieve("Wildfire conditions California", params)
print("Keys:", list(data_res.keys())[:10])
print("n retrieved:", len(data_res.get("retrieved_data") or []))
if data_res.get("error"):
    print("error:", data_res["error"][:200])


## 7. PredictionReasoningAgent — `predict`

Có checkpoint SeasFire trong config → model thật; không thì heuristic / fallback (xem log).


In [ ]:
from calm.utils.time_utils import resolve_time_range

pred_params = {
    "location": "California Central Valley",
    "latitude": 36.5,
    "longitude": -119.5,
    "time_range": resolve_time_range("next 7 days", default_today=True),
    "met_data": {
        "temperature": 28.0,
        "humidity": 40.0,
        "wind_speed": 12.0,
        "precipitation": 0.0,
    },
    "spatial_data": {"ndvi_mean": 0.35},
}

pred_out = orch.prediction_agent.predict(pred_params)
print("risk_level:", pred_out.get("risk_level"))
print("confidence:", pred_out.get("confidence"))
print("used_fallback:", pred_out.get("used_fallback"))
if pred_out.get("error"):
    print("error:", pred_out["error"][:200])


## 8. RSENModule — `validate`

Kiểm tra Plausible / Implausible so với met + spatial (gọi LLM).


In [ ]:
val = orch.rsen.validate(
    prediction={"risk_level": pred_out.get("risk_level", "Medium"), "confidence": pred_out.get("confidence", 0.5)},
    met_data=pred_params["met_data"],
    spatial_data=pred_params["spatial_data"],
)
print("validation_decision:", val.get("validation_decision", val.get("decision")))
print("rationale (trunc):", (val.get("final_rationale") or "")[:200])


## 9. WildfireQAAgent — `invoke`

Dùng dữ liệu đã retrieve (có thể rỗng) để trả lời có citations.


In [ ]:
qa_res = orch.qa_agent.invoke(
    "List two main natural causes of wildfires.",
    pre_retrieved=data_res if data_res.get("retrieved_data") else {"retrieved_data": []},
)
fo = qa_res.get("final_output") or {}
print("answer (trunc):", (fo.get("answer") or "")[:300])
print("citations:", len(fo.get("citations") or []))
print("approved:", qa_res.get("approved"))


## 10. ExecutionAgent — `execute_step`

Mô phỏng một bước plan: `data_knowledge` → retrieve.


In [ ]:
step = {
    "step_id": "s1",
    "agent": "data_knowledge",
    "action": "retrieve environmental data",
    "parameters": {"location": "Amazon basin"},
    "prompt": "Wildfire drivers Amazon",
}

ctx = {
    "query": "Wildfire drivers Amazon",
    "original_query": "Wildfire drivers Amazon",
    "parameters": {"location": "Amazon basin"},
}

ex_res = orch.executor.execute_step(step, ctx)
print("executor keys:", list(ex_res.keys())[:8])
print("retrieved len:", len((ex_res.get("retrieved_data") or [])))


## 11. FeedbackRefinerAgent

Điều chỉnh confidence / action dựa trên quality + RSEN (không train lại model).


In [ ]:
from calm.agents.feedback_refiner_agent import FeedbackRefinerAgent

refiner = FeedbackRefinerAgent(memory_store=memory_store, config={})
refined = refiner.run(
    prediction_output=pred_out,
    rsen_outputs=val,
    data_quality={"level": "medium", "sources_count": 1},
)
print("calibrated_confidence:", refined.get("calibrated_confidence"))
print("final_decision:", refined.get("final_decision"))
print("actions:", refined.get("refinement_actions", [])[:3])


## 12. EvaluatorAgent — LLM-as-a-Judge

Chấm response của orchestrator (5 tiêu chí).


In [ ]:
from calm.agents.evaluator_agent import EvaluatorAgent

evaluator = EvaluatorAgent(llm=llm, config={"passing_score": 75.0})
orch_snapshot = orch.run("What is fuel moisture in wildfire context?")
ev = evaluator.evaluate(response=orch_snapshot, query="What is fuel moisture in wildfire context?")
print("average_score:", ev.get("average_score"))
print("passed:", ev.get("passed"))
print("scores:", ev.get("scores"))


## 13. Intent hints (routing fallback)


In [ ]:
from calm.utils.intent_hints import infer_task_from_keywords

assert infer_task_from_keywords("What are risks of fire?") == "qa"
assert infer_task_from_keywords("Predict wildfire risk next 7 days") == "prediction"
print("intent_hints OK")


## 14. SafetyChecker

Kiểm tra hành động trước khi gọi tool (dùng trong ExecutionAgent / tools).


In [ ]:
from calm.tools.safety_check import SafetyChecker

safety = SafetyChecker(llm=llm)
ok = safety.is_safe("Retrieve ERA5 summary for a bounding box in California for research.")
print("safe (benign action):", ok)


## 15. CALMOrchestrator — luồng hoàn chỉnh

`run(query)` → normalize → plan → route → QA **hoặc** prediction pipeline → memory.


In [ ]:
def print_orch_result(label: str, r: dict) -> None:
    tt = r.get("task_type")
    print(f"\n=== {label} ===")
    print("task_type:", tt)
    print("error:", r.get("error"))
    if tt == "qa":
        print("answer (trunc):", (r.get("answer") or "")[:280])
    else:
        print("risk_level:", r.get("risk_level"), "| decision:", r.get("decision"))
        print("rationale (trunc):", (r.get("rationale") or "")[:200])
    mem = r.get("memory_reflection") or {}
    print("memory stored:", mem.get("stored"), mem.get("reason") or mem.get("mode") or "")


r_qa = orch.run("What are common human causes of wildfires?")
print_orch_result("QA pipeline", r_qa)

r_pr = orch.run("Predict wildfire risk for California Central Valley over the next 7 days")
print_orch_result("Prediction pipeline", r_pr)

print("\nDone — full pipeline smoke test.")
